1. Task A: Masked Language Modeling (The "Fill-in-the-blank" Test)
This demonstrates BERT's Bidirectional nature. It looks at the words after the mask to guess what the masked word should be.

In [3]:
from transformers import pipeline

# Load the fill-mask pipeline with the standard BERT model
bert_unmasker = pipeline('fill-mask', model='bert-base-uncased')

# A sentence with a [MASK] token
text = "The student with USN 1RVU23CSE414 is working in the [MASK] lab."

# Ask BERT to predict the missing word
results = bert_unmasker(text)

print(f"Original Text: {text}\n")
print("BERT's Top Predictions:")
for result in results:
    print(f"-> Word: '{result['token_str']}' | Confidence: {result['score']:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original Text: The student with USN 1RVU23CSE414 is working in the [MASK] lab.

BERT's Top Predictions:
-> Word: 'same' | Confidence: 0.1528
-> Word: 'chemistry' | Confidence: 0.0770
-> Word: 'physics' | Confidence: 0.0644
-> Word: 'biology' | Confidence: 0.0421
-> Word: 'computer' | Confidence: 0.0359


2. Task B: Extracting Semantic Embeddings
Here, we use BERT to convert text into a 768-dimensional vector (the "Hidden State"). This is the "Encoder" part of the BERT architecture.

In [4]:
import torch
from transformers import BertTokenizer, BertModel

# 1. Initialize Tokenizer and Model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_vector(sentence):
    # Tokenize: BERT needs [CLS] at start and [SEP] at end
    inputs = tokenizer(sentence, return_tensors='pt')

    # Forward pass through BERT layers
    with torch.no_grad():
        outputs = model(**inputs)

    # The 'last_hidden_state' contains the numerical representation
    # of every token. We take the [CLS] token (index 0) as the
    # summary of the whole sentence.
    sentence_embedding = outputs.last_hidden_state[0][0]
    return sentence_embedding

# Example Usage
sentence = "Satyatma is implementing BERT for his AI lab report."
vector = get_bert_vector(sentence)

print(f"Sentence: '{sentence}'")
print(f"Vector Shape: {vector.shape}")
print(f"First 5 values of the vector: {vector[:5].numpy()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence: 'Satyatma is implementing BERT for his AI lab report.'
Vector Shape: torch.Size([768])
First 5 values of the vector: [-0.3834469   0.00732016  0.08396322 -0.2126539  -0.46128753]


Bidirectionality: Unlike an RNN (which reads left to right), the code in Task A shows that BERT can "see" the word lab at the end of the sentence to help it guess that the missing word is computer, research, or physics.

Tokenization: Notice the BertTokenizer. BERT doesn't just split by spaces; it uses WordPiece tokenization. If it sees an unknown word like "Chincholi," it might split it into "Chin" and "##choli" so it can still process the parts it recognizes.

The [CLS] Token: In Task B, we extracted the vector from index 0. This is the Classification (CLS) token. BERT is designed so that this specific token absorbs the context of the entire sentence, making it perfect for sentiment analysis or document classification.